# Training a Franken model

This tutorial walks through training a Franken interatomic potential: loading data, configuring the model, and automatically finding suitable training settings.

We use water structures with reference energies and forces from density functional theory (DFT) calculations at the RPBE+D3 level, collected by [Montero de Hijes et al.](https://doi.org/10.1063/5.0197105).

Run the cells in order, locally or on [Google Colab](https://colab.research.google.com/github/CSML-IIT-UCL/franken/blob/main/notebooks/training.ipynb). The first cell installs Franken with MACE support if needed. Internet access is required in the first run to download the data and pretrained model.

All configuration options are listed in the [API reference](https://franken.readthedocs.io/reference/franken-api/franken.config.html).

In [1]:
try:
    import franken
except ImportError:
    %pip install "franken[mace]"
    import franken

In [2]:
import json

import ase.io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from franken.autotune import autotune
from franken.backbones.utils import CacheDir
from franken.config import (
    AutotuneConfig,
    DatasetConfig,
    GaussianRFConfig,
    HPSearchConfig,
    MaceBackboneConfig,
    SolverConfig,
)
from franken.datasets.registry import DATASET_REGISTRY

## Load the data

To optimize the franken model we need two datasets:

- The **training set** provides reference energies and forces for fitting the model.
- The **validation set** contains structures held out of fitting, used to compare training settings and select the best model.

Franken reads structures through the Atomic Simulation Environment (ASE), for example from extended XYZ files (`.xyz` or `.extxyz`). Each frame should contain a structure with its total energy and atomic forces. Use Å for positions, eV for energies, and eV/Å for forces. Optional stress labels use eV/Å³; this tutorial trains only on energies and forces.

The cell below retrieves the water dataset from Franken's registry, downloading it to `CacheDir.get()` if needed, and displays the first frame of each split. For your own data, pass your file paths to `DatasetConfig` below.

In [3]:
# Locate the datasets and inspect the first structure in each split.
train_path = DATASET_REGISTRY.get_path("water", "train", base_path=CacheDir.get())
val_path = DATASET_REGISTRY.get_path("water", "val", base_path=CacheDir.get())

print(f"Train path: {train_path}")
train_atoms = ase.io.read(train_path, index=0)
print(f"Train[0]: {train_atoms}")

print(f"\nValidation path: {val_path}")
val_atoms = ase.io.read(val_path, index=0)
print(f"Validation[0]: {val_atoms}")

Train path: /home/lbonati@iit.local/.franken/water/ML_AB_dataset_1.xyz
Train[0]: Atoms(symbols='H128O64', pbc=True, cell=[[13.101991, -1.0387e-05, -1.4377e-05], [0.0, 13.101991, 1.1531e-05], [0.0, 0.0, 13.101992]], calculator=SinglePointCalculator(...))

Validation path: /home/lbonati@iit.local/.franken/water/ML_AB_dataset_2-val.xyz
Validation[0]: Atoms(symbols='H128O64', pbc=True, cell=[[13.290949802, 2.2628e-05, 5.59e-06], [0.0, 13.290950224, -8.5923e-05], [0.0, -0.0, 13.290956559]], calculator=SinglePointCalculator(...))


`DatasetConfig` specifies the data paths. For this tutorial, `max_train_samples=8` randomly selects eight training structures, while the full validation set is used. A seed can be passed for reproducibility (default: `1337`). 

In [4]:
dataset_cfg = DatasetConfig(
    train_path=str(train_path),  # Replace with your own path, e.g. "train.xyz".
    val_path=str(val_path),
    max_train_samples=8,
)

## Franken's ingredients

Franken combines three components:

1. A **pretrained backbone** represents each atom's local environment as numerical features.
2. A **kernel**, approximated by random features, relates these environments to their energies.
3. A **linear solver** fits the weights used to predict energies (forces follow from energy derivatives with respect to atomic positions).

Only the linear weights are fitted; the backbone and random feature map stay fixed. Settings such as the kernel length scale and regularization strength are **hyperparameters**, which we will choose using validation errors.

### 1) Backbone

The backbone is a graph neural network (GNN) pretrained on atomistic data. Franken reuses its learned (=data-driven) description of local chemical environments.

Here we use MACE's `mace_mh/0` model. `path_or_id` accepts a registered model ID or a compatible local checkpoint. See the [backbone documentation](https://franken.readthedocs.io/topics/training_backbones.html) or run `franken.backbones list` for other choices. Registered models are downloaded when first needed.

> **Cache location:** Datasets and backbone GNN checkpoints are stored by default in `$HOME/.franken`. To change this, set `FRANKEN_CACHE_DIR` before running the import and data-loading cells:
>
> ```python
> import os
>
> os.environ["FRANKEN_CACHE_DIR"] = "/path/to/my/franken-cache"
> ```

In [5]:
gnn_config = MaceBackboneConfig(path_or_id="mace_mh/0")

### 2) Kernel and random features

A **Gaussian kernel** measures similarity between atomic environments using their backbone features. Random Fourier features (RFs) approximate this kernel with a finite feature map that can be fitted by a linear solver.

`GaussianRFConfig` has two main settings:

- **`num_random_features`** sets the feature count and number of fitted weights. More features improve the kernel approximation but require more memory and computation. We have found a few thousands to work fine. Here we use `2048`. 
- **`length_scale`** sets the resolution of the kernel (how quickly similarity decreases with distance in the normalized feature space). To find this parameter we can perform a grid-search (`HPSearchConfig(values=[...])`) which tries the five listed length scales and selects using the metrics on the validation set.

> Note: This is expensive since it requires a different training per each value. You can narrow the search on a representative training subset, then use a fixed value such as `length_scale=10.0` for a larger run.

In [6]:
rf_config = GaussianRFConfig(
    num_random_features=2048,
    length_scale=HPSearchConfig(values=[1.0, 5.0, 10.0, 20.0, 30.0]),
)

> **Alternative: combine length scales.**  The grid search can be avoided by using a  `MultiscaleGaussianRFConfig`, which combines Random Features associated with different lengthscales in a single model, avoiding a separate fit for each scale. To use it, you can replace `rf_config` above with:
>
> ```python
> from franken.config import MultiscaleGaussianRFConfig
>
> rf_config = MultiscaleGaussianRFConfig(
>     num_random_features=2048,
>     length_scale_low=4.0,
>     length_scale_high=24.0,
>     length_scale_num=4,
> )
> ```
>
> This uses 2048 features in total across four evenly spaced scales between 4 and 24. 

### 3) Solver

The solver fits the linear weights by minimizing weighted squared energy and force errors, plus an L2 penalty. This is **regularized least squares**, also called ridge regression.

| Setting | Role | Default search |
| --- | --- | --- |
| `l2_penalty` | Discourages large weights to limit overfitting. | Six values from $10^{-11}$ to $10^{-6}$. |
| `force_weight` | Controls the emphasis on force errors relative to energy errors. | Ten values from $10^{-2}$ to $10^4$. |

The energy weight defaults to `1.0`, so `force_weight` gives the force-to-energy weight ratio. Target weights are then normalized to 1.

**Solver searches are relatively inexpensive:** optimizing the solver parameters do not require recomputing the features and their derivatives, making the search cheap.

In [7]:
solver_cfg = SolverConfig(
    l2_penalty=HPSearchConfig(start=-11, stop=-6, num=6, scale="log"),
    force_weight=HPSearchConfig(start=-2, stop=4, num=10, scale="log"),
)

### Hyperparameter optimization with `autotune`

`AutotuneConfig` combines the settings above. Calling `autotune` then:

1. Loads the data and backbone, scales features, and estimates baseline atomic energies from the training set.
2. Computes features and derivatives for each length scale, then fits each solver combination.
3. Evaluates the candidates, selects the best model using validation errors, and saves the results.

Our grid contains **300 candidates**: five length scales × 60 solver combinations.

- **`run_dir`** sets the parent directory for results. Each call creates a unique folder and returns its path as `run_path`.
- **`jac_chunk_size`** controls how many derivative directions are processed together within a structure. Leave it at `"auto"`, or try `16` and decrease it if you encounter CUDA out-of-memory errors. Smaller chunks reduce memory use but may be slower.
- **`metrics`** lists the errors to report. Available metrics are MAE and RMSE. 
- **`best_model_selection`** sets the validation metrics used to rank candidates, defaulting here to both energy and force MAE. It does not change the fitting objective but only how the best model is selected; see the [metrics documentation](https://franken.readthedocs.io/topics/training_metrics.html).

> **Command-line equivalent:**
> ```bash
> franken.autotune \
>     --train-path "$HOME/.franken/water/ML_AB_dataset_1.xyz" \
>     --val-path "$HOME/.franken/water/ML_AB_dataset_2-val.xyz" \
>     --max-train-samples 8 \
>     --backbone mace --mace.path-or-id "mace_mh/0" \
>     --rf gaussian --gaussian.num-rf 4096 \
>     --gaussian.length-scale "[1.0, 5.0, 10.0, 20.0, 30.0]" \
>     --l2-penalty "(-11, -6, 6, log)" \
>     --force-weight "(-2, 4, 10, log)" \
>     --metrics energy_MAE forces_MAE \
>     --jac-chunk-size auto \
>     --run-dir "./results"
> ```

> **Custom workflows:** For more control, you can also use `FrankenPotential` with `LowMemRandomFeaturesTrainer` or `RandomFeaturesTrainer`; see the [API reference](https://franken.readthedocs.io/reference/index.html).

In [ ]:
autotune_cfg = AutotuneConfig(
    dataset=dataset_cfg,
    solver=solver_cfg,
    backbone=gnn_config,
    rfs=rf_config,
    metrics=["energy_MAE", "forces_MAE"],
    jac_chunk_size="auto",  # Use a smaller integer if memory is limited.
    run_dir="./results",
)

run_path = autotune(autotune_cfg)

atomic_energies: None
console_logging_level: INFO
dtype: float64
eval_splits: None
jac_chunk_size: auto
rf_normalization: leading_eig
run_dir: ./results
save_every_model: False
save_fmaps: False
scale_by_species: True
seed: 1337
backbone:
    family: mace
    interaction_block: 2
    path_or_id: mace_mh/0
best_model_selection:
    []
dataset:
    max_train_samples: 8
    name: null
    test_path: null
    train_path: /home/lbonati@iit.local/.franken/water/ML_AB_dataset_1.xyz
    val_path: /home/lbonati@iit.local/.franken/water/ML_AB_dataset_2-val.xyz
metrics:
    - energy_MAE
    - forces_MAE
rfs:
    length_scale:
      num: null
      scale: null
      start: null
      stop: null
      value: null
      values:
      - 1.0
      - 5.0
      - 10.0
      - 20.0
      - 30.0
    num_random_features: 2048
    rf_type: gaussian
    rng_seed: 1337
    use_offset: true
solver:
    energy_weight: 1.0
    force_weight:
      num: 10
      scale: log
      start: -2
      stop: 4
      value

ASE -> Franken (val): 100%|██████████| 189/189 [00:00<00:00, 298.71it/s]


Computing dataset statistics:   0%|          | 0/8 [00:00<?, ?it/s]

2026-09-07 22:54:27.228 INFO (rank 0): jacobian chunk size automatically set to 16


### Inspect outputs

The results are stored in run_path, which represents a unique folder called `run_DATE_TIME_...` inside `run_dir` containing:

| File | Contents |
| --- | --- |
| `best_ckpt.pt` | Selected model checkpoint. |
| `best.json` | Its hyperparameters, metrics, and fitting timings. |
| `configs.json` | Run configuration and hardware information. |
| `log.json` | Results from the hyperparameter search. |

The next cell displays `best.json`. Compare the `train` and `validation` metrics: a large gap can indicate overfitting or differences between datasets. `hyperparameters` records the selected settings, while `timings` reports parts of fitting in seconds.

For a final evaluation, you can also supply an independent test set through `DatasetConfig.test_path`. To make predictions with the saved checkpoint, use Franken's [ASE calculator](https://franken.readthedocs.io/topics/interface_ase.html).

In [ ]:
!cat {run_path}/best.json

{
    "checkpoint": {
        "hash": "da0f2e7ae388565ceb5285ac83c05492",
        "rf_weight_id": 8
    },
    "timings": {
        "cov_coeffs": 85.60891724284738,
        "solve": 0.0672919424250722
    },
    "metrics": {
        "train": {
            "energy_MAE": 0.141870751732325,
            "forces_MAE": 7.581024348735809
        },
        "validation": {
            "energy_MAE": 0.17251996059897418,
            "forces_MAE": 12.350173087347121
        }
    },
    "hyperparameters": {
        "franken": {
            "path_or_id": "mace_mh/0",
            "family": "mace",
            "interaction_block": 2
        },
        "random_features": {
            "rf_type": "gaussian",
            "num_random_features": 4096,
            "length_scale": 20.0,
            "use_offset": true,
            "rng_seed": 1337
        },
        "input_scaler": {
            "scale_by_Z": true
        },
        "solver": {
            "energy_weight": 1.0,
            "forces_weight": 